
###### 11_vector_search

###### Purpose

Create a Databricks Vector Search endpoint and Delta Sync index over the precomputed customer-note embeddings, then validate semantic retrieval using a natural-language query.


###### Technologies Used

- Databricks

- Delta Lake

- Databricks SDK

- PySpark

- Unity Catalog

- Databricks Vector Search

- Databricks Embedding Foundation Model (databricks-gte-large-en)

###### Input

-  Customer-note embeddings Delta table

-  Unique customer_id primary key

-  Precomputed embedding column

-  Vector Search endpoint and index configuration

-  Embedding model for query vectors


######  Output


-  Databricks Vector Search endpoint

-  Triggered Delta Sync index

-  Synchronized customer-note embeddings

-  Top-K semantically relevant customer notes


######  Architecture

```text

Customer Notes Embeddings Delta Table
        ↓
Enable Change Data Feed
        ↓
Vector Search Endpoint
        ↓
Triggered Delta Sync Index
       ↓
Synchronize Index
        ↓
Natural-Language Question
        ↓
Generate Query Embedding
        ↓
Similarity Search
        ↓
Top-k Similar Customer notes
     
```


###### Section 0 Install Vector Search Client

In [0]:
#Install Vector Search client
%pip install databricks-vectorsearch
dbutils.library.restartPython()

###### Section 1 : Load Project Configuration

In [0]:
%run ./00_project_config

###### Section 2 : Import Libraries and Initialize Clients

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
vsc = VectorSearchClient()

###### Section 3 : Enable Change Data Feed

In [0]:
spark.sql(f"""
    ALTER TABLE {EMBEDDINGS_TABLE}
    SET TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    )
""")

print("Change Data Feed enabled.")

###### Section 4 : Validate Source Embeddings Table

In [0]:
source_df = spark.table(EMBEDDINGS_TABLE)

total_count = source_df.count()

if total_count == 0:
    raise ValueError(
        f"The source table {EMBEDDINGS_TABLE} is empty."
    )

distinct_count = (
    source_df
    .select("customer_id")
    .distinct()
    .count()
)

if total_count != distinct_count:
    raise ValueError(
        "customer_id must be unique because it is used as the primary key."
    )

print(f"Total rows: {total_count}")
print(f"Distinct customer IDs: {distinct_count}")

# verify null primary keys:
null_key_count = (
    source_df
    .filter("customer_id IS NULL")
    .count()
)

if null_key_count > 0:
    raise ValueError(
        f"Found {null_key_count} null customer IDs."
    )


###### Section 5 : Determine Embedding Dimension

In [0]:
from pyspark.sql.functions import size, col

invalid_embedding_count = (
    source_df
    .filter(
        col("embedding").isNull()
        | (size("embedding") == 0)
    )
    .count()
)

if invalid_embedding_count > 0:
    raise ValueError(
        f"Found {invalid_embedding_count} invalid embeddings."
    )

dimension_rows = (
    source_df
    .select(size("embedding").alias("dimension"))
    .distinct()
    .collect()
)

dimensions = [
    row["dimension"]
    for row in dimension_rows
]

if len(dimensions) != 1:
    raise ValueError(
        f"Inconsistent embedding dimensions found: {dimensions}"
    )

embedding_dimension = dimensions[0]

print("Embedding dimension:", embedding_dimension)

###### Section 6 : Create or Retrieve Vector Search Endpoint

In [0]:
# Section 6: Create or Retrieve Vector Search Endpoint

from datetime import timedelta

try:
    endpoint = vsc.get_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME
    )

    print(
        f"Vector Search endpoint already exists: "
        f"{VECTOR_SEARCH_ENDPOINT_NAME}"
    )

except Exception as exc:
    # For this learning notebook, retrieval failure is treated
    # as a missing endpoint. In production, catch the specific
    # resource-not-found exception so permission errors are not hidden.
    print("Endpoint retrieval failed:", exc)
    print(
        f"Creating Vector Search endpoint: "
        f"{VECTOR_SEARCH_ENDPOINT_NAME}"
    )

    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )

# Wait until the endpoint is ready before creating an index.
vsc.wait_for_endpoint(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    timeout=timedelta(minutes=10)
)

endpoint = vsc.get_endpoint(
    name=VECTOR_SEARCH_ENDPOINT_NAME
)

print("Vector Search endpoint is ready.")
print(endpoint)

###### Section 7 : Create or Retrieve Delta Sync Index

In [0]:
try:
# Section 7: Create or Retrieve Delta Sync Index

try:
    index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_INDEX_NAME
    )

    print(
        f"Delta Sync index already exists: "
        f"{VECTOR_INDEX_NAME}"
    )

except Exception as exc:
    # For this learning notebook, retrieval failure is treated
    # as a missing index. In production, catch the specific
    # resource-not-found exception.
    print("Index retrieval failed:", exc)
    print(
        f"Creating Delta Sync index: "
        f"{VECTOR_INDEX_NAME}"
    )

    index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        source_table_name=EMBEDDINGS_TABLE,
        index_name=VECTOR_INDEX_NAME,
        pipeline_type="TRIGGERED",
        primary_key="customer_id",
        embedding_dimension=embedding_dimension,
        embedding_vector_column="embedding",
        columns_to_sync=[
            "customer_id",
            "note"
        ]
    )

    print("Delta Sync index creation started.")

# Retrieve the index again so the same variable is used
# whether it was newly created or already existed.
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=VECTOR_INDEX_NAME
)

print("Delta Sync index retrieved successfully.")
print(index.describe())

#vsc.delete_index(
    #endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    #index_name=VECTOR_INDEX_NAME)


###### Section 8 : Trigger and Monitor Index Synchronization

In [0]:
# Section 8: Trigger and Monitor Index Synchronization

from datetime import timedelta

print("Starting triggered index synchronization...")

index.sync()

# Wait until the index is online and the triggered update completes.
index.wait_until_ready(
    verbose=True,
    timeout=timedelta(minutes=10),
    wait_for_updates=True
)

index_status = index.describe()

detailed_state = (
    index_status
    .get("status", {})
    .get("detailed_state", "UNKNOWN")
)

print("=" * 60)
print("Index synchronization completed.")
print(f"Index name  : {VECTOR_INDEX_NAME}")
print(f"Index state : {detailed_state}")
print("=" * 60)

if not detailed_state.startswith("ONLINE"):
    raise RuntimeError(
        "The Vector Search index did not reach an ONLINE state. "
        f"Current state: {detailed_state}"
    )

###### Section 9  : Create Semantic Search Function

In [0]:
def semantic_search(question: str, num_results: int = 3):
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    if not response.data or response.data[0].embedding is None:
        raise ValueError(
            "The embedding model returned no query embedding."
        )

    question_embedding = [
        float(value)
        for value in response.data[0].embedding
    ]

    if len(question_embedding) != embedding_dimension:
        raise ValueError(
            "Query embedding dimension does not match "
            "the indexed embedding dimension."
        )

    return index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

###### Section 10 : Run Similarity Search Test

In [0]:
question = "Which customers are likely to cancel service?"

search_results = semantic_search(
    question=question,
    num_results=3
)

print("Question:")
print(question)

print("\nRaw Vector Search result:")
print(search_results)

###### Notebook Summary

- Validated the customer-note embeddings source table.

- Created or retrieved a Databricks Vector Search endpoint.

- Created or retrieved a triggered Delta Sync index.

- Synchronized and monitored the index until it became available.

- Generated query embeddings using the same embedding model used for the stored notes.

- Retrieved the Top-K semantically relevant customer notes.


###### Key Learnings

-  Vector Search compares numerical embeddings rather than raw text.

-  The endpoint provides the serving infrastructure, while the Delta Sync index contains and searches the indexed data.

-  Change Data Feed enables incremental synchronization from the source Delta table.

-  A Triggered index is updated when index.sync() is called.

-  The query and stored notes must use compatible embeddings in the same vector space.

- Primary-key values must be unique and non-null.

-  Semantic search can retrieve related text even when the question uses different words.

-  Only columns included in the index can be returned in search results.

###### Notebook Conclusion

- In this notebook, we created a semantic-retrieval layer by building a Databricks Vector Search endpoint and a triggered Delta Sync index over the customer-note embedding table.

- We synchronized the index, generated a vector for a user question, and retrieved the most semantically relevant customer notes.

- In the next notebook, these retrieved notes will become grounded context for a Retrieval-Augmented Generation application.

###### Next Notebook

12_RAG

- The purpose of this notebook is to combine Vector Search with a Large Language Model (LLM) to answer user questions using retrieved Customer notes information.